# Advanced Problems with Solutions: Python Import Variants and Misconceptions

This notebook contains advanced practice problems on Python import variants, namespace binding, `sys.modules`, aliasing, wildcard imports, import-time misconceptions, and import performance.

Each problem includes a complete solution.

## Setup

Run this cell first. It creates helper utilities used throughout the notebook.

In [1]:
import importlib
import math
import sys
import timeit
from types import ModuleType

def show_names(*names, namespace=None):
    namespace = globals() if namespace is None else namespace
    return {name: name in namespace for name in names}

def unload(module_name):
    sys.modules.pop(module_name, None)

def module_status(module_name, namespace=None):
    namespace = globals() if namespace is None else namespace
    return {
        'in_sys_modules': module_name in sys.modules,
        'in_namespace': module_name in namespace,
        'sys_modules_object': repr(sys.modules.get(module_name))
    }

## Problem 1 — Prove that `from module import name` still loads the module

A common misconception is that:

```python
from cmath import exp
```

loads only the `exp` function and not the whole `cmath` module.

Tasks:

1. Remove `cmath` from `sys.modules` if it is already loaded.
2. Use `from cmath import exp`.
3. Show that `exp` exists in the current namespace.
4. Show that `cmath` does not exist in the current namespace.
5. Show that `cmath` exists in `sys.modules`.
6. Retrieve the actual `cmath` module object from `sys.modules` and use another function from it.

### Solution 1

In [2]:
globals().pop('cmath', None)
globals().pop('exp', None)
unload('cmath')

print('Before import:', module_status('cmath'))

from cmath import exp

print('After import:', module_status('cmath'))
print('exp in globals:', 'exp' in globals())
print('cmath in globals:', 'cmath' in globals())

cmath_from_cache = sys.modules['cmath']
print('Retrieved module:', cmath_from_cache)
print('Using another function from cached module:', cmath_from_cache.sqrt(1 + 1j))

Before import: {'in_sys_modules': False, 'in_namespace': False, 'sys_modules_object': 'None'}
After import: {'in_sys_modules': True, 'in_namespace': False, 'sys_modules_object': "<module 'cmath' (built-in)>"}
exp in globals: True
cmath in globals: False
Retrieved module: <module 'cmath' (built-in)>
Using another function from cached module: (1.09868411346781+0.45508986056222733j)


## Problem 2 — Compare namespace effects of import variants

Write a function `run_import_experiment(statement)` that executes an import statement in a fresh temporary namespace and returns:

- the names created in that namespace,
- whether the imported module is in `sys.modules`,
- whether the module name itself was bound in the temporary namespace.

Test these statements:

```python
import math
import math as r_math
from math import sqrt
from math import sqrt as r_sqrt
from math import *
```

### Solution 2

In [3]:
def run_import_experiment(statement):
    namespace = {'__builtins__': __builtins__}
    before = set(namespace)

    exec(statement, namespace)

    after = set(namespace)
    created = sorted(after - before)

    return {
        'statement': statement,
        'created_names': created,
        'math_in_sys_modules': 'math' in sys.modules,
        'math_bound_in_namespace': 'math' in namespace
    }

statements = [
    'import math',
    'import math as r_math',
    'from math import sqrt',
    'from math import sqrt as r_sqrt',
    'from math import *'
]

for statement in statements:
    print(run_import_experiment(statement))

{'statement': 'import math', 'created_names': ['math'], 'math_in_sys_modules': True, 'math_bound_in_namespace': True}
{'statement': 'import math as r_math', 'created_names': ['r_math'], 'math_in_sys_modules': True, 'math_bound_in_namespace': False}
{'statement': 'from math import sqrt', 'created_names': ['sqrt'], 'math_in_sys_modules': True, 'math_bound_in_namespace': False}
{'statement': 'from math import sqrt as r_sqrt', 'created_names': ['r_sqrt'], 'math_in_sys_modules': True, 'math_bound_in_namespace': False}
{'statement': 'from math import *', 'created_names': ['acos', 'acosh', 'asin', 'asinh', 'atan', 'atan2', 'atanh', 'cbrt', 'ceil', 'comb', 'copysign', 'cos', 'cosh', 'degrees', 'dist', 'e', 'erf', 'erfc', 'exp', 'exp2', 'expm1', 'fabs', 'factorial', 'floor', 'fma', 'fmod', 'frexp', 'fsum', 'gamma', 'gcd', 'hypot', 'inf', 'isclose', 'isfinite', 'isinf', 'isnan', 'isqrt', 'lcm', 'ldexp', 'lgamma', 'log', 'log10', 'log1p', 'log2', 'modf', 'nan', 'nextafter', 'perm', 'pi', 'pow', '

## Problem 3 — Reconstruct import statements using `importlib`

For each import statement below, write the equivalent operation using `importlib.import_module` and normal assignment:

```python
import math
import math as r_math
from math import sqrt
from math import sqrt as r_sqrt
```

### Solution 3

In [4]:
namespace = {}

namespace['math'] = importlib.import_module('math')
namespace['r_math'] = importlib.import_module('math')
namespace['sqrt'] = importlib.import_module('math').sqrt
namespace['r_sqrt'] = importlib.import_module('math').sqrt

print(namespace['math'])
print(namespace['r_math'])
print(namespace['sqrt'])
print(namespace['r_sqrt'])
print(namespace['math'] is namespace['r_math'])
print(namespace['sqrt'] is namespace['r_sqrt'])

<module 'math' (built-in)>
<module 'math' (built-in)>
<built-in function sqrt>
<built-in function sqrt>
True
True


## Problem 4 — Diagnose name clobbering

The following code silently replaces one `sqrt` function with another:

```python
from cmath import sqrt
from math import sqrt
```

Tasks:

1. Prove that the second import overwrites the first name binding.
2. Show why this can break complex-number code.
3. Rewrite the imports safely using aliases.

### Solution 4

In [5]:
from cmath import sqrt
first_sqrt = sqrt
print('After from cmath import sqrt:', sqrt)

from math import sqrt
second_sqrt = sqrt
print('After from math import sqrt:', sqrt)

print('Was sqrt overwritten?', first_sqrt is not second_sqrt)

try:
    print(sqrt(2 + 2j))
except TypeError as exc:
    print('Problem caused by clobbering:', exc)

from math import sqrt as real_sqrt
from cmath import sqrt as complex_sqrt

print('real_sqrt(2):', real_sqrt(2))
print('complex_sqrt(2 + 2j):', complex_sqrt(2 + 2j))

After from cmath import sqrt: <built-in function sqrt>
After from math import sqrt: <built-in function sqrt>
Was sqrt overwritten? True
Problem caused by clobbering: must be real number, not complex
real_sqrt(2): 1.4142135623730951
complex_sqrt(2 + 2j): (1.5537739740300374+0.6435942529055826j)


## Problem 5 — Demonstrate wildcard import risks with a custom module

Create an in-memory simulation of a module namespace that exports these names:

```python
sqrt = 'fake sqrt'
pi = 'fake pi'
_private = 'hidden'
__all__ = ['sqrt', 'pi']
```

Tasks:

1. Simulate what `from module import *` would inject.
2. Show that exported names can overwrite existing names.
3. Explain why `__all__` matters.

### Solution 5

In [6]:
fake_module = ModuleType('fake_module')
fake_module.sqrt = 'fake sqrt'
fake_module.pi = 'fake pi'
fake_module._private = 'hidden'
fake_module.__all__ = ['sqrt', 'pi']

namespace = {'sqrt': math.sqrt, 'existing_name': 'keep me'}

def simulate_star_import(module, namespace):
    exported_names = getattr(module, '__all__', None)

    if exported_names is None:
        exported_names = [name for name in vars(module) if not name.startswith('_')]

    for name in exported_names:
        namespace[name] = getattr(module, name)

    return namespace

print('Before:', namespace)
simulate_star_import(fake_module, namespace)
print('After:', namespace)
print('_private imported?', '_private' in namespace)

Before: {'sqrt': <built-in function sqrt>, 'existing_name': 'keep me'}
After: {'sqrt': 'fake sqrt', 'existing_name': 'keep me', 'pi': 'fake pi'}
_private imported? False


## Problem 6 — Show that local imports do not reload modules every time

Some people think this function reloads `math` on every call:

```python
def f(x):
    import math
    return math.sqrt(x)
```

Tasks:

1. Remove `math` from `sys.modules`.
2. Call a function that imports `math` locally.
3. Show that subsequent calls reuse the same cached module object from `sys.modules`.
4. Explain the real overhead.

### Solution 6

In [7]:
unload('math')

def local_import_sqrt(x):
    import math
    return math.sqrt(x), id(math)

result_1, module_id_1 = local_import_sqrt(25)
result_2, module_id_2 = local_import_sqrt(36)

print('First result:', result_1)
print('Second result:', result_2)
print('Same module object reused:', module_id_1 == module_id_2)
print('math in sys.modules:', 'math' in sys.modules)
print('Cached object id:', id(sys.modules['math']))

import math

First result: 5.0
Second result: 6.0
Same module object reused: True
math in sys.modules: True
Cached object id: 2799433632400


The module is not fully reloaded each time. After the first import, Python mostly performs a lookup in `sys.modules` and binds the local name. The main reason to avoid repeated local imports is usually readability and dependency visibility, not catastrophic performance.

## Problem 7 — Benchmark import and attribute-access variants

Benchmark these approaches:

1. `math.sqrt(x)` with `math` imported globally.
2. `sqrt(x)` with `sqrt` imported directly.
3. A function that imports `math` locally each time.
4. A function that uses a default argument to bind `math.sqrt` once.

Your goal is not just to find the fastest version. Your goal is to compare relative and absolute differences.

### Solution 7

In [8]:
setup_code = """
import math
from math import sqrt

def local_import(x):
    import math
    return math.sqrt(x)

def bound_default(x, sqrt=math.sqrt):
    return sqrt(x)
"""

tests = {
    'global module attribute': 'math.sqrt(2)',
    'direct imported symbol': 'sqrt(2)',
    'local import inside function': 'local_import(2)',
    'bound default argument': 'bound_default(2)'
}

results = {}
for label, stmt in tests.items():
    elapsed = timeit.timeit(stmt, setup=setup_code, number=1_000_000)
    results[label] = elapsed

baseline = results['global module attribute']

for label, elapsed in results.items():
    print(f'{label:32} {elapsed:.6f}s  diff_vs_baseline={elapsed - baseline:+.6f}s')

global module attribute          0.111106s  diff_vs_baseline=+0.000000s
direct imported symbol           0.074454s  diff_vs_baseline=-0.036651s
local import inside function     0.227683s  diff_vs_baseline=+0.116578s
bound default argument           0.108327s  diff_vs_baseline=-0.002779s


The directly imported symbol is often faster because it avoids repeated attribute lookup on the module object. However, in most real programs the absolute difference is tiny unless this occurs inside a very hot loop. Readability and clarity usually matter more.

## Problem 8 — Identify import style tradeoffs

For each snippet, explain the tradeoff and provide a better version when appropriate.

### Snippet A

```python
from math import *
from cmath import *
result = sqrt(-1)
```

### Snippet B

```python
def area(radius):
    import math
    return math.pi * radius ** 2
```

### Snippet C

```python
from statistics import mean
print(mean([1, 2, 3]))
```

### Snippet D

```python
import numpy as np
```

### Solution 8

### Snippet A

Problem: wildcard imports make it unclear which `sqrt` is active. The second wildcard import may overwrite names from the first.

Better version:

```python
import math
import cmath

real_result = math.sqrt(4)
complex_result = cmath.sqrt(-1)
```

Or:

```python
from math import sqrt as real_sqrt
from cmath import sqrt as complex_sqrt
```

### Snippet B

This does not reload `math` every time after the first call, because `math` is cached in `sys.modules`. However, putting imports at the top of the file is usually clearer.

Better version:

```python
import math

def area(radius):
    return math.pi * radius ** 2
```

A local import may be reasonable when an import is expensive, optional, or needed only in a rarely used code path.

### Snippet C

This is usually fine. `mean` is descriptive and unlikely to collide accidentally in a small module. Direct imports can improve readability when only a few names are needed.

### Snippet D

This is a standard community alias. It improves readability because `np` is widely understood in Python data-science code. Aliases are best when they are conventional or when they remove genuine verbosity without hiding meaning.

## Problem 9 — Build a namespace collision detector

Write a function:

```python
def would_star_import_clobber(module, namespace):
    ...
```

It should return the exported names from `module` that already exist in `namespace`.

Use it to check whether `from math import *` would overwrite names in a namespace that already contains:

```python
{'sqrt': 'custom sqrt', 'pi': 3, 'my_name': 'safe'}
```

### Solution 9

In [9]:
def exported_names(module):
    names = getattr(module, '__all__', None)
    if names is not None:
        return list(names)
    return [name for name in dir(module) if not name.startswith('_')]

def would_star_import_clobber(module, namespace):
    exported = exported_names(module)
    return sorted(name for name in exported if name in namespace)

namespace = {'sqrt': 'custom sqrt', 'pi': 3, 'my_name': 'safe'}

collisions = would_star_import_clobber(math, namespace)
print('Collisions:', collisions)

for name in collisions:
    print(f'{name!r}: before={namespace[name]!r}, after={getattr(math, name)!r}')

Collisions: ['pi', 'sqrt']
'pi': before=3, after=3.141592653589793
'sqrt': before='custom sqrt', after=<built-in function sqrt>


## Problem 10 — Explain package nuance

For simple modules, `from module import name` loads the module and then binds `name` in the current namespace.

Packages can be more nuanced because packages may contain submodules and subpackages.

Tasks:

1. Import `xml`.
2. Check whether `xml.etree` is automatically loaded.
3. Import `xml.etree.ElementTree`.
4. Check which related names now appear in `sys.modules`.
5. Explain the difference between importing a package and importing one of its submodules.

### Solution 10

In [10]:
for name in list(sys.modules):
    if name == 'xml' or name.startswith('xml.etree'):
        sys.modules.pop(name, None)

import xml

print('After import xml:')
print('xml in sys.modules:', 'xml' in sys.modules)
print('xml.etree in sys.modules:', 'xml.etree' in sys.modules)
print('xml.etree.ElementTree in sys.modules:', 'xml.etree.ElementTree' in sys.modules)

import xml.etree.ElementTree as ET

print('\nAfter import xml.etree.ElementTree as ET:')
for name in sorted(name for name in sys.modules if name == 'xml' or name.startswith('xml.etree')):
    print(name)

root = ET.Element('root')
print('Created XML element:', root.tag)

After import xml:
xml in sys.modules: True
xml.etree in sys.modules: False
xml.etree.ElementTree in sys.modules: False

After import xml.etree.ElementTree as ET:
xml
xml.etree
xml.etree.ElementPath
xml.etree.ElementTree
Created XML element: root


Importing a package does not necessarily import every submodule in that package. A package can be a namespace that later allows Python to locate submodules. This is different from a simple module such as `math` or `cmath`, where importing one name from the module still loads that module object.

## Summary

Key takeaways:

1. Import variants mainly control which names are bound in the current namespace.
2. The imported module is still cached in `sys.modules`.
3. `from module import name` does not bind the module name itself in your namespace.
4. Wildcard imports can overwrite existing names and reduce readability.
5. Aliases are useful when they are clear, conventional, or prevent collisions.
6. Repeated imports usually do not reload modules; they normally reuse the `sys.modules` cache.
7. Micro-optimizing import style is rarely worth sacrificing clarity.
8. Packages can behave differently from simple modules because submodules are loaded separately.